In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.datasets import load_diabetes, load_breast_cancer, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split

from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import (
    BaggingClassifier,
    RandomForestClassifier,
    AdaBoostClassifier
)

from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    silhouette_score,
    adjusted_rand_score
)

RANDOM_STATE = 42
OUTPUT_DIR = Path.cwd() / "question_3_results"

print("Libraries imported successfully.")

Libraries imported successfully.


In [ ]:
data = load_diabetes(as_frame=True)

X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    random_state=RANDOM_STATE
)

regression_models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.01, max_iter=20000)
}

regression_rows = []

for name, estimator in regression_models.items():
    model = Pipeline([
        ("scale", StandardScaler()),
        ("model", estimator)
    ])

    model.fit(X_train, y_train)
    prediction = model.predict(X_test)

    regression_rows.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, prediction),
        "MSE": mean_squared_error(y_test, prediction),
        "RMSE": np.sqrt(mean_squared_error(y_test, prediction)),
        "R2": r2_score(y_test, prediction)
    })

regression_results = pd.DataFrame(regression_rows)

regression_results = regression_results.sort_values(
    "RMSE"
).reset_index(drop=True)

print("REGRESSION PERFORMANCE")
print(regression_results.round(4).to_string(index=False))

REGRESSION PERFORMANCE
            Model     MAE       MSE    RMSE     R2
 Ridge Regression 42.8120 2892.0146 53.7775 0.4541
 Lasso Regression 42.7950 2898.3680 53.8365 0.4529
Linear Regression 42.7941 2900.1936 53.8534 0.4526


In [ ]:
data = load_breast_cancer(as_frame=True)

X_train, X_test, y_train, y_test = train_test_split(
    data.data,
    data.target,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=data.target
)

classification_models = {
    "Decision Tree (Gini)": DecisionTreeClassifier(
        criterion="gini",
        max_depth=5,
        random_state=RANDOM_STATE
    ),

    "Decision Tree (Entropy)": DecisionTreeClassifier(
        criterion="entropy",
        max_depth=5,
        random_state=RANDOM_STATE
    ),

    "Logistic Regression (L2/Ridge)": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l2",
            solver="lbfgs",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),

    "Logistic Regression (L1/Lasso)": Pipeline([
        ("scale", StandardScaler()),
        ("model", LogisticRegression(
            penalty="l1",
            solver="liblinear",
            max_iter=5000,
            random_state=RANDOM_STATE
        ))
    ]),

    "K-Nearest Neighbours": Pipeline([
        ("scale", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=5))
    ]),

    "Gaussian Naive Bayes": GaussianNB(),

    "Bagging": BaggingClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=200,
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        random_state=RANDOM_STATE
    )
}

classification_rows = []

for name, model in classification_models.items():
    model.fit(X_train, y_train)

    prediction = model.predict(X_test)
    probability = model.predict_proba(X_test)[:, 1]

    classification_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, prediction),
        "Precision": precision_score(y_test, prediction),
        "Recall": recall_score(y_test, prediction),
        "F1": f1_score(y_test, prediction),
        "ROC-AUC": roc_auc_score(y_test, probability)
    })

classification_results = pd.DataFrame(classification_rows)

classification_results = classification_results.sort_values(
    "F1",
    ascending=False
).reset_index(drop=True)

print("CLASSIFICATION PERFORMANCE")
print(classification_results.round(4).to_string(index=False))

CLASSIFICATION PERFORMANCE
                         Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression (L1/Lasso)    0.9912     0.9863  1.0000 0.9931   0.9967
Logistic Regression (L2/Ridge)    0.9825     0.9861  0.9861 0.9861   0.9954
                      AdaBoost    0.9561     0.9467  0.9861 0.9660   0.9818
                 Random Forest    0.9561     0.9589  0.9722 0.9655   0.9931
          K-Nearest Neighbours    0.9561     0.9589  0.9722 0.9655   0.9788
          Gaussian Naive Bayes    0.9386     0.9452  0.9583 0.9517   0.9878
                       Bagging    0.9386     0.9452  0.9583 0.9517   0.9927
       Decision Tree (Entropy)    0.9298     0.9706  0.9167 0.9429   0.9435
          Decision Tree (Gini)    0.9211     0.9565  0.9167 0.9362   0.9163


In [ ]:
data = load_iris(as_frame=True)

X = StandardScaler().fit_transform(data.data)
y = data.target

clustering_models = {
    "K-Means": KMeans(
        n_clusters=3,
        n_init=10,
        random_state=RANDOM_STATE
    ),

    "Hierarchical (Ward)": AgglomerativeClustering(
        n_clusters=3,
        linkage="ward"
    ),

    "Hierarchical (Complete)": AgglomerativeClustering(
        n_clusters=3,
        linkage="complete"
    ),

    "Gaussian Mixture": GaussianMixture(
        n_components=3,
        n_init=10,
        random_state=RANDOM_STATE
    )
}

clustering_rows = []

for name, model in clustering_models.items():
    labels = model.fit_predict(X)

    clustering_rows.append({
        "Model": name,
        "Silhouette": silhouette_score(X, labels),
        "Adjusted Rand Index": adjusted_rand_score(y, labels),
        "Number of clusters": len(np.unique(labels))
    })

clustering_results = pd.DataFrame(clustering_rows)

clustering_results = clustering_results.sort_values(
    "Silhouette",
    ascending=False
).reset_index(drop=True)

print("CLUSTERING PERFORMANCE")
print(clustering_results.round(4).to_string(index=False))

CLUSTERING PERFORMANCE
                  Model  Silhouette  Adjusted Rand Index  Number of clusters
                K-Means      0.4599               0.6201                   3
Hierarchical (Complete)      0.4496               0.5726                   3
    Hierarchical (Ward)      0.4467               0.6153                   3
       Gaussian Mixture      0.3742               0.9039                   3


In [ ]:
data = load_breast_cancer(as_frame=True)

X_scaled = StandardScaler().fit_transform(data.data)

pca = PCA(n_components=2)
reduced = pca.fit_transform(X_scaled)

reconstructed = pca.inverse_transform(reduced)

pca_results = pd.DataFrame([
    {
        "Method": "PCA (2 components)",
        "Original features": X_scaled.shape[1],
        "Reduced features": reduced.shape[1],
        "Explained variance": pca.explained_variance_ratio_.sum(),
        "Reconstruction MSE": mean_squared_error(
            X_scaled,
            reconstructed
        )
    }
])

print("PCA PERFORMANCE")
print(pca_results.round(4).to_string(index=False))

PCA PERFORMANCE
            Method  Original features  Reduced features  Explained variance  Reconstruction MSE
PCA (2 components)                 30                 2              0.6324              0.3676


In [ ]:
def save_visualizations(
    regression: pd.DataFrame,
    classification: pd.DataFrame,
    clustering: pd.DataFrame,
) -> None:

    OUTPUT_DIR.mkdir(exist_ok=True)

    figure, axes = plt.subplots(1, 3, figsize=(18, 5))

    axes[0].bar(
        regression["Model"],
        regression["RMSE"],
        color="steelblue"
    )
    axes[0].set_title("Regression: RMSE (lower is better)")
    axes[0].tick_params(axis="x", rotation=70)

    axes[1].bar(
        classification["Model"],
        classification["F1"],
        color="darkorange"
    )
    axes[1].set_title("Classification: F1 (higher is better)")
    axes[1].tick_params(axis="x", rotation=70)

    axes[2].bar(
        clustering["Model"],
        clustering["Silhouette"],
        color="seagreen"
    )
    axes[2].set_title("Clustering: Silhouette (higher is better)")
    axes[2].tick_params(axis="x", rotation=70)

    figure.tight_layout()
    figure.savefig(
        OUTPUT_DIR / "model_comparison.png",
        dpi=150
    )
    plt.close(figure)

In [ ]:
def main():

    regression = regression_results
    classification = classification_results
    clustering = clustering_results
    pca = pca_results

    OUTPUT_DIR.mkdir(exist_ok=True)

    regression.to_csv(
        OUTPUT_DIR / "regression_metrics.csv",
        index=False
    )

    classification.to_csv(
        OUTPUT_DIR / "classification_metrics.csv",
        index=False
    )

    clustering.to_csv(
        OUTPUT_DIR / "clustering_metrics.csv",
        index=False
    )

    pca.to_csv(
        OUTPUT_DIR / "pca_metrics.csv",
        index=False
    )

    save_visualizations(
        regression,
        classification,
        clustering
    )

    print("\nREGRESSION PERFORMANCE")
    print(regression.round(4).to_string(index=False))

    print("\nCLASSIFICATION PERFORMANCE")
    print(classification.round(4).to_string(index=False))

    print("\nCLUSTERING PERFORMANCE")
    print(clustering.round(4).to_string(index=False))

    print("\nPCA PERFORMANCE")
    print(pca.round(4).to_string(index=False))

    print(f"\nResults saved to: {OUTPUT_DIR}")


main()


REGRESSION PERFORMANCE
            Model     MAE       MSE    RMSE     R2
 Ridge Regression 42.8120 2892.0146 53.7775 0.4541
 Lasso Regression 42.7950 2898.3680 53.8365 0.4529
Linear Regression 42.7941 2900.1936 53.8534 0.4526

CLASSIFICATION PERFORMANCE
                         Model  Accuracy  Precision  Recall     F1  ROC-AUC
Logistic Regression (L1/Lasso)    0.9912     0.9863  1.0000 0.9931   0.9967
Logistic Regression (L2/Ridge)    0.9825     0.9861  0.9861 0.9861   0.9954
                      AdaBoost    0.9561     0.9467  0.9861 0.9660   0.9818
                 Random Forest    0.9561     0.9589  0.9722 0.9655   0.9931
          K-Nearest Neighbours    0.9561     0.9589  0.9722 0.9655   0.9788
          Gaussian Naive Bayes    0.9386     0.9452  0.9583 0.9517   0.9878
                       Bagging    0.9386     0.9452  0.9583 0.9517   0.9927
       Decision Tree (Entropy)    0.9298     0.9706  0.9167 0.9429   0.9435
          Decision Tree (Gini)    0.9211     0.9565  0.9167 